# 06 外部因素统计关联分析

本 Notebook 对应阶段 4：问题三建模。目标是分析天气、节假日、活动日等因素与销量之间的统计关联，不直接证明因果关系。

## 代码 1：读取建模基础表

这段代码解决“外部因素字段是否真实存在”的问题。特别注意：原始附件没有湿度字段，所以后续不分析湿度。

In [ ]:
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
df = pd.read_csv(ROOT / 'data/processed/modeling_base_table.csv')
df['date'] = pd.to_datetime(df['date'])
external_cols = ['weather','max_temperature','min_temperature','wind_power','is_holiday','is_weekend','is_activity_day','weekday','month','store_id','product_id','category']
df[external_cols + ['positive_sales']].head()


输出理解：如果上述字段存在，就可以进入问题三变量构造。`humidity` 不在字段中，不能补造湿度变量。

## 代码 2：构造日总销量表

这段代码解决描述性分析的粒度问题。由于天气、节假日、活动日是日期层面的变量，先把所有门店商品汇总成日总销量，再比较不同外部因素下的均值。

In [ ]:
daily = pd.read_csv(ROOT / 'tables/q3_daily_external_factor_table.csv')
daily.head()


输出理解：每一行是一日总销量及该日外部因素。这个表用于天气、节假日、周末、活动日的均值比较。

## 代码 3：查看天气描述性统计

这段代码解决“不同天气下销量是否有差异”的问题，只是描述性比较，不控制混杂因素。

In [ ]:
weather_stats = pd.read_csv(ROOT / 'tables/q3_weather_descriptive_stats.csv')
weather_stats.head(10)


输出理解：`mean_daily_sales` 是该天气下平均日销量。样本天数少的天气结论更不稳定，不能过度解释。

## 代码 4：读取控制变量回归结果

这段代码解决“控制门店、商品、星期、月份后，外部因素是否仍有关联”的问题。回归系数表示统计关联，不表示因果影响。

In [ ]:
reg = pd.read_csv(ROOT / 'tables/q3_regression_coefficients_external.csv')
reg[['factor','variable','coef','p_value','direction','comparable_abs_effect']].head(15)


输出理解：`coef` 为控制变量后的回归系数。连续变量的大小需结合四分位距折算；天气系数是相对基准天气的差异。

## 代码 5：读取随机森林置换重要性

这段代码解决“非线性模型下哪些特征对预测更重要”的问题。置换重要性表示打乱某个特征后模型误差上升多少，不能说明销量变化方向。

In [ ]:
rf_imp = pd.read_csv(ROOT / 'tables/q3_random_forest_permutation_importance.csv')
rf_imp.head(12)


输出理解：重要性越高，说明该特征对随机森林验证集预测贡献越大。门店、商品等控制变量可能排名较高，这反映基础需求差异。

## 代码 6：读取统计关联强度排序

这段代码解决论文最终如何排序的问题。排序综合考虑回归可比效应和随机森林重要性，并明确稳定性与局限。

In [ ]:
summary = pd.read_csv(ROOT / 'tables/q3_factor_summary.csv')
summary


输出理解：该表可直接作为问题三“统计关联强度排序”的依据。论文中应使用“关联”表述，不写“导致”。